# สำรวจข้อมูล 2/4: แนวโน้มและความสัมพันธ์ (Trends / Correlation)

| ไฟล์ | เกณฑ์ 3.5.2 (18 คะแนน) |
|---|---|
| ✅ eda_step1_descriptive | 3.5.2.1 สถิติเบื้องต้น (8) |
| **→** eda_step2_trends | 3.5.2.2 แนวโน้ม/ความสัมพันธ์ (5) |
|  eda_step3_outliers_missing | 3.5.2.3 ค่าผิดปกติ/การขาดหาย (3) |
|  eda_step4_insights | 3.5.2.4 Insight + ตอบคำถามจาก step1_questions (2) |

ตอบเกณฑ์ 3.5.2.2 (5 คะแนน) ครบ 4 ข้อ:
1. **แนวโน้มเวลา**: line + rolling mean (ข้อมูลรายปี จึงไม่มีฤดูกาล)
2. **ความสัมพันธ์เชิงตัวเลข**: heatmap + วิเคราะห์ทิศทางและความแรง
3. **เปรียบเทียบข้ามกลุ่ม**: boxplot แยกทวีป / อาเซียน
4. **ปฏิสัมพันธ์ (แบ่งชั้น)**: ดูความสัมพันธ์แยกตามกลุ่มย่อย (facet)

> รัน **Kernel → Restart Kernel and Run All Cells**

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

DATA = os.path.join("final", "energy_clean.csv")
if not os.path.exists(DATA):
    raise FileNotFoundError("ไม่พบ final/energy_clean.csv ให้รันขั้นทำความสะอาด 1-5 ให้สำเร็จก่อน")
OUT = "eda_output"                       # เก็บรูปและตารางไว้ใช้ทำสไลด์
os.makedirs(OUT, exist_ok=True)

df = pd.read_csv(DATA)
LATEST = int(df.loc[df["year_complete"], "year"].max())   # ปีล่าสุดที่ข้อมูลครบ = 2021
y = df[df["year"] == LATEST].copy()                        # ภาพตัดขวาง (ทุกประเทศ ปีเดียว)
ASEAN = sorted(df.loc[df["is_asean"], "country"].unique())
FUEL_COLOR = {"coal": "#4A4039", "oil": "#8C5A3C", "gas": "#E0913A", "nuclear": "#7B61A8", "hydro": "#2E78AD",
              "wind": "#5FB3A1", "solar": "#E8B923", "biofuel": "#7A9A3A", "other_renewables": "#9AA5A8"}

def savefig(name):
    plt.savefig(os.path.join(OUT, name), dpi=150, bbox_inches="tight")

print(f"ข้อมูล: {df.shape[0]:,} แถว x {df.shape[1]} คอลัมน์ | {df['country'].nunique()} ประเทศ | ปี {df['year'].min()}-{df['year'].max()}")
print(f"ปีที่ใช้วิเคราะห์ภาพตัดขวาง: {LATEST} ({len(y)} ประเทศ)")

## 2.1 แนวโน้มตามเวลา
### โลก: ไฟฟ้าทั้งหมด และสัดส่วนพลังงานหมุนเวียน
ใช้เฉพาะปีที่ข้อมูลครบ (`year_complete`) เพราะปี 2022 ประเทศยังรายงานไม่ครบ ยอดรวมจะดูเหมือนลดลงผิด ๆ
**Rolling mean 3 ปี** = ค่าเฉลี่ยเคลื่อนที่ ช่วยลดความผันผวนรายปีให้เห็นทิศทางชัดขึ้น

In [ ]:
w = df[df["year_complete"]].groupby("year")[["electricity_generation", "renewables_electricity",
                                              "fossil_electricity", "solar_electricity", "wind_electricity", "coal_electricity"]].sum()
w["renew_share"] = w["renewables_electricity"] / w["electricity_generation"] * 100
w["renew_share_roll3"] = w["renew_share"].rolling(3, center=True).mean()

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].plot(w.index, w["electricity_generation"] / 1000, marker="o", color="#1C2733")
ax[0].set_title("World electricity generation (thousand TWh)"); ax[0].set_ylim(0, None)
ax[1].plot(w.index, w["renew_share"], marker="o", alpha=0.5, label="yearly")
ax[1].plot(w.index, w["renew_share_roll3"], lw=3, color="#5FB3A1", label="3-year rolling mean")
ax[1].set_title("World renewables share of electricity (%)"); ax[1].set_ylim(0, None); ax[1].legend()
plt.tight_layout(); savefig("trend_world.png"); plt.show()

first, last = w.index.min(), w.index.max()
print(f"ไฟฟ้าโลก {first}->{last}: {w.loc[first, 'electricity_generation']:,.0f} -> {w.loc[last, 'electricity_generation']:,.0f} TWh "
      f"(+{(w.loc[last, 'electricity_generation'] / w.loc[first, 'electricity_generation'] - 1) * 100:.0f}%)")
for a, b in [(2000, 2010), (2010, LATEST)]:
    print(f"สัดส่วนหมุนเวียน {a}->{b}: {w.loc[a, 'renew_share']:.1f}% -> {w.loc[b, 'renew_share']:.1f}% "
          f"(+{w.loc[b, 'renew_share'] - w.loc[a, 'renew_share']:.1f} จุด ใน {b - a} ปี)")

**เห็นอะไร:** ไฟฟ้าโลกเพิ่มขึ้นเกือบทุกปี (ยกเว้นปี 2009 วิกฤตการเงิน และปี 2020 โควิด) ส่วนสัดส่วนหมุนเวียน **แทบไม่ขยับในทศวรรษแรก** (2000–2010) แล้ว **เร่งตัวชัดหลังปี 2010**
**หมายความว่า:** การเปลี่ยนผ่านพลังงานเพิ่งเริ่มเร่งในช่วง 10 ปีหลัง พร้อมกับแสงอาทิตย์และลมที่ราคาถูกลง
**ทำอะไรต่อ:** ดูว่าไทยเร่งตามโลกทันหรือไม่ (ข้อถัดไป)

In [ ]:
# ไทย: ความต้องการไฟฟ้า อัตราโตรายปี และสัดส่วนเชื้อเพลิง
th = df[df["country"] == "Thailand"].set_index("year").sort_index()
th["demand_yoy"] = th["electricity_demand"].pct_change() * 100

fig, ax = plt.subplots(1, 3, figsize=(17, 4))
ax[0].plot(th.index, th["electricity_demand"], marker="o", color="#1C2733", label="demand")
ax[0].plot(th.index, th["electricity_generation"], marker="o", color="#1F5F8B", label="generation")
ax[0].set_title("Thailand: demand vs generation (TWh)"); ax[0].set_ylim(0, None); ax[0].legend()
ax[1].bar(th.index, th["demand_yoy"], color=["#B3412E" if v < 0 else "#9AA5A8" for v in th["demand_yoy"].fillna(0)])
ax[1].plot(th.index, th["demand_yoy"].rolling(3, center=True).mean(), color="#1F5F8B", lw=2.5, label="3-year rolling mean")
ax[1].axhline(0, color="black", lw=0.8); ax[1].set_title("Thailand: demand growth per year (%)"); ax[1].legend()
for f in ["gas", "coal", "hydro", "biofuel", "solar", "wind"]:
    ax[2].plot(th.index, th[f"{f}_share_elec"], label=f, color=FUEL_COLOR[f], lw=2.5 if f == "gas" else 1.8)
ax[2].set_title("Thailand: share of electricity by fuel (%)"); ax[2].legend(ncol=3, fontsize=8); ax[2].set_ylim(0, 100)
plt.tight_layout(); savefig("trend_thailand.png"); plt.show()

g1 = th["demand_yoy"].loc[2001:2010].mean(); g2 = th["demand_yoy"].loc[2011:LATEST].mean()
print(f"ความต้องการไฟฟ้าไทยโตเฉลี่ย {g1:.1f}%/ปี (2001-2010) และ {g2:.1f}%/ปี (2011-{LATEST})")
print(f"ก๊าซ: {th.loc[2000, 'gas_share_elec']:.1f}% (2000) -> สูงสุด {th['gas_share_elec'].max():.1f}% (ปี {th['gas_share_elec'].idxmax()}) -> {th.loc[LATEST, 'gas_share_elec']:.1f}% ({LATEST})")

**เห็นอะไร:** ความต้องการไฟฟ้าไทยโตเร็วในทศวรรษแรก แล้ว **ชะลอลง** ในทศวรรษหลัง (ปี 2020 ติดลบจากโควิด) ส่วน **ก๊าซครองสัดส่วนสูงตลอด** แต่เริ่มลดลงช่วงหลัง ขณะที่ชีวมวลและแสงอาทิตย์ค่อย ๆ เพิ่ม
**หมายความว่า:** ไทยกำลังกระจายเชื้อเพลิงออกจากก๊าซ แต่ช้า
**ทำอะไรต่อ:** เทียบกับอาเซียน (ข้อ 2.3) และหาว่าอะไรสัมพันธ์กับสัดส่วนหมุนเวียน (ข้อ 2.2)

## 2.2 ความสัมพันธ์ระหว่างตัวแปร (Correlation heatmap)
ใช้ **Spearman** แทน Pearson เพราะข้อมูลเบ้มาก (จากไฟล์ที่ 1) Spearman ดูจาก **อันดับ** จึงไม่ถูกค่าสุดโต่งดึง

| ค่า r | ความแรง |
|---|---|
| 0.0–0.2 | แทบไม่สัมพันธ์ |
| 0.2–0.4 | อ่อน |
| 0.4–0.7 | ปานกลาง |
| > 0.7 | แรง |

ค่าบวก = ไปทางเดียวกัน · ค่าลบ = สวนทางกัน

In [ ]:
y["gdp_per_capita"] = y["gdp"] / y["population"]
y2018 = df[df["year"] == 2018].assign(gdp_per_capita=lambda d: d["gdp"] / d["population"])   # GDP มีถึงปี 2018
VARS = {"per_capita_electricity": "elec/person", "energy_per_capita": "energy/person", "gdp_per_capita": "GDP/person",
        "renewables_share_elec": "renewables %", "gas_share_elec": "gas %", "coal_share_elec": "coal %",
        "hydro_share_elec": "hydro %", "carbon_intensity_elec": "carbon intensity"}
base = y2018[~y2018["low_precision"]][list(VARS)].rename(columns=VARS)
corr = base.corr(method="spearman")

fig, ax = plt.subplots(figsize=(8.5, 7))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=45, ha="right"); ax.set_yticks(range(len(corr)), corr.index)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=9,
                color="white" if abs(corr.iloc[i, j]) > 0.6 else "black")
plt.colorbar(im, fraction=0.046); ax.set_title(f"Spearman correlation, 2018 ({len(base)} countries)")
plt.tight_layout(); savefig("corr_heatmap.png"); plt.show()

pairs = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack().sort_values(key=abs, ascending=False)
pairs.head(8).round(2).to_frame("r (เรียงตามความแรง)")

**อ่านผล (ใช้ปี 2018 เพราะต้องมี GDP):**
- **ไฟฟ้าต่อคน ↔ GDP ต่อคน: บวกแรงมาก** ประเทศรวยใช้ไฟมาก ข้อนี้คาดได้
- **หมุนเวียน % ↔ ความเข้มคาร์บอน: ลบแรงมาก** ใช้หมุนเวียนมาก ไฟฟ้ายิ่งสะอาด (เป็นนิยามโดยตรง จึงใช้ยืนยันว่าข้อมูลสอดคล้องกัน)
- **หมุนเวียน % ↔ พลังน้ำ %: บวกแรง** หมุนเวียนของโลกส่วนใหญ่ยังเป็นพลังน้ำ ไม่ใช่แสงอาทิตย์หรือลม
- **หมุนเวียน % ↔ GDP ต่อคน / ไฟฟ้าต่อคน: ลบอ่อน (r ≈ −0.2)** ประเทศรวยไม่ได้ใช้ไฟฟ้าสะอาดกว่า กลับเอนไปทางตรงข้ามเล็กน้อย ความรวยจึงไม่ได้ทำให้ไฟฟ้าสะอาดขึ้นเอง

> ⚠️ สหสัมพันธ์ไม่ได้บอกเหตุและผล เช่น GDP กับไฟฟ้าต่อคนอาจโตไปด้วยกันเพราะการพัฒนาอุตสาหกรรม

In [ ]:
# คู่ตัวแปรสำคัญ: ไฟฟ้าต่อคน vs GDP ต่อคน (สเกล log ทั้งสองแกน)
d = y2018[~y2018["low_precision"]].dropna(subset=["gdp_per_capita", "per_capita_electricity"])
d = d[d["per_capita_electricity"] > 0]
lx, ly = np.log10(d["gdp_per_capita"]), np.log10(d["per_capita_electricity"])
slope, intercept = np.polyfit(lx, ly, 1)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(d["gdp_per_capita"], d["per_capita_electricity"], s=18, alpha=0.5, color="#9AA5A8")
a = d[d["is_asean"]]
ax.scatter(a["gdp_per_capita"], a["per_capita_electricity"], s=40, color="#1F5F8B")
for _, r in a.iterrows():
    ax.annotate(r["country"], (r["gdp_per_capita"], r["per_capita_electricity"]), fontsize=8, xytext=(4, 2), textcoords="offset points")
xs = np.linspace(lx.min(), lx.max(), 50)
ax.plot(10 ** xs, 10 ** (intercept + slope * xs), color="#B3412E", lw=2, label=f"trend: +1% GDP -> +{slope:.2f}% electricity")
ax.set_xscale("log"); ax.set_yscale("log"); ax.legend()
ax.set_xlabel("GDP per person (international-$, log)"); ax.set_ylabel("Electricity per person (kWh, log)")
ax.set_title(f"Richer countries use more electricity (2018, n={len(d)}, ASEAN in blue)")
plt.tight_layout(); savefig("scatter_gdp_elec.png"); plt.show()
print(f"ความชันบนสเกล log = {slope:.2f} → GDP ต่อคนสูงขึ้น 1% ไฟฟ้าต่อคนสูงขึ้นราว {slope:.2f}%")

## 2.3 เปรียบเทียบข้ามกลุ่ม: ทวีป และอาเซียน

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
regions = y.groupby("region")["renewables_share_elec"].median().sort_values().index
data = [y.loc[y["region"] == r, "renewables_share_elec"].dropna() for r in regions]
ax[0].boxplot(data, vert=False); ax[0].set_yticks(range(1, len(regions) + 1), regions)
ax[0].scatter([y.loc[y["country"] == "Thailand", "renewables_share_elec"].iloc[0]], [list(regions).index("Asia") + 1],
              color="#1F5F8B", s=80, zorder=3, label="Thailand")
ax[0].set_title(f"Renewables share by region, {LATEST} (%)"); ax[0].legend()

a = df[df["is_asean"] & df["year"].isin([2010, LATEST])].pivot(index="country", columns="year", values="gas_share_elec")
a = a.sort_values(LATEST)
ax[1].barh(a.index, a[2010], color="#E0913A", alpha=0.35, label="2010")
ax[1].barh(a.index, a[LATEST], color="#E0913A", height=0.45, label=str(LATEST))
ax[1].set_title("ASEAN: gas share of electricity (%)"); ax[1].legend()
for lbl in ax[1].get_yticklabels():
    if lbl.get_text() == "Thailand": lbl.set_fontweight("bold"); lbl.set_color("#1F5F8B")
plt.tight_layout(); savefig("compare_groups.png"); plt.show()

print(y.groupby("region")["renewables_share_elec"].agg(["median", "count"]).round(1).sort_values("median", ascending=False))
print("\nสัดส่วนก๊าซ อาเซียน", LATEST, ":"); print(a[LATEST].sort_values(ascending=False).round(1).to_string())

**เห็นอะไร:** อเมริกาใต้มีมัธยฐานสัดส่วนหมุนเวียนสูงสุด (เขื่อนใหญ่ เช่น บราซิล ปารากวัย) ส่วน **เอเชียต่ำสุด** ในอาเซียน ไทยใช้ก๊าซ **สูงเป็นอันดับ 2 รองจากสิงคโปร์** ซึ่งเป็นเกาะที่ไม่มีทางเลือกอื่น ขณะที่เวียดนามลดก๊าซจาก 49% เหลือ 11% ใน 11 ปี
**หมายความว่า:** ภูมิศาสตร์ (แม่น้ำ ภูเขา) กำหนดสัดส่วนหมุนเวียนมาก และไทยพึ่งก๊าซสูงผิดปกติแม้เทียบเพื่อนบ้านที่มีทรัพยากรคล้ายกัน
**ทำอะไรต่อ:** ศึกษาว่าเวียดนามเปลี่ยนอย่างไร (ข้อ 2.4)

## 2.4 ปฏิสัมพันธ์: แบ่งชั้นตามช่วงเวลาและกลุ่มประเทศ (Facet)
คำถาม: **ประเทศที่สัดส่วนหมุนเวียนเพิ่มเร็ว เพิ่มจากเชื้อเพลิงอะไร?** และ **กลุ่มรายได้ต่างกันเปลี่ยนต่างกันไหม?**
แบ่งประเทศเป็น 3 กลุ่มตามไฟฟ้าต่อคน (ใช้ไฟน้อย / กลาง / มาก) แล้วดูการเปลี่ยนแปลงแยกตามช่วงเวลา

In [ ]:
p = df[df["year"].isin([2010, LATEST]) & ~df["low_precision"]].pivot_table(
    index="country", columns="year", values=["solar_share_elec", "wind_share_elec", "hydro_share_elec", "per_capita_electricity"])
p = p.dropna()
p["use_group"] = pd.qcut(p[("per_capita_electricity", 2010)], 3, labels=["low use", "mid use", "high use"])
change = pd.DataFrame({f: p[(f"{f}_share_elec", LATEST)] - p[(f"{f}_share_elec", 2010)] for f in ["solar", "wind", "hydro"]})
change["use_group"] = p["use_group"]
tab = change.groupby("use_group", observed=True).median().round(2)

fig, ax = plt.subplots(1, 3, figsize=(15, 3.8), sharey=True)
for i, f in enumerate(["solar", "wind", "hydro"]):
    ax[i].bar(tab.index.astype(str), tab[f], color=FUEL_COLOR[f]); ax[i].axhline(0, color="black", lw=0.8)
    ax[i].set_title(f"{f}: median change in share 2010->{LATEST} (pts)")
plt.tight_layout(); savefig("facet_change_by_group.png"); plt.show()
tab

In [ ]:
# ASEAN แยกทีละประเทศ (small multiples): เปลี่ยนจากเชื้อเพลิงไหนไปเชื้อเพลิงไหน
fig, axes = plt.subplots(2, 5, figsize=(18, 6), sharex=True, sharey=True)
for ax, c in zip(axes.flat, ASEAN):
    d = df[df["country"] == c].set_index("year").loc[:LATEST]
    for f in ["gas", "coal", "hydro", "solar", "oil"]:
        ax.plot(d.index, d[f"{f}_share_elec"], color=FUEL_COLOR[f], lw=2.2 if f == "gas" else 1.5, label=f)
    ax.set_title(c, fontweight="bold" if c == "Thailand" else "normal", color="#1F5F8B" if c == "Thailand" else "black")
    ax.set_ylim(0, 100)
axes[0, 0].legend(fontsize=8)
plt.suptitle("ASEAN: share of electricity by fuel (%)", fontsize=14); plt.tight_layout()
savefig("facet_asean.png"); plt.show()

**เห็นอะไร:**
- กลุ่มที่ **ใช้ไฟมาก** เพิ่มสัดส่วนลมและแสงอาทิตย์ได้มากกว่ากลุ่มที่ใช้ไฟน้อย ส่วนพลังน้ำแทบไม่เพิ่มในทุกกลุ่ม (เขื่อนใหม่สร้างยาก)
- ในอาเซียน **เวียดนาม** เพิ่มแสงอาทิตย์ก้าวกระโดด จากเกือบ 0% (2018) เป็น 10.5% (2021) ใน 3 ปี ส่วน **ไทย** เส้นก๊าซลดลงช้า ๆ

**หมายความว่า:** การเติบโตของพลังงานสะอาดยุคใหม่มาจาก **แสงอาทิตย์และลม** ไม่ใช่เขื่อน และเพื่อนบ้านที่ระดับใกล้ไทยทำได้เร็วกว่า
**ทำอะไรต่อ:** ใช้เป็นข้อเสนอแนะในไฟล์ที่ 4 และแสดงในแดชบอร์ดหน้าเปรียบเทียบ

## ✅ สรุปขั้นนี้ (เกณฑ์ 3.5.2.2)
| ข้อย่อย | หลักฐาน |
|---|---|
| 3.5.2.2.1 แนวโน้มเวลา + rolling mean | ข้อ 2.1 `trend_world.png`, `trend_thailand.png` |
| 3.5.2.2.2 heatmap + ทิศทาง/ความแรง | ข้อ 2.2 `corr_heatmap.png`, `scatter_gdp_elec.png` |
| 3.5.2.2.3 เปรียบเทียบข้ามกลุ่ม | ข้อ 2.3 `compare_groups.png` |
| 3.5.2.2.4 ปฏิสัมพันธ์/แบ่งชั้น (facet) | ข้อ 2.4 `facet_change_by_group.png`, `facet_asean.png` |

**ไฟล์ถัดไป:** `eda_step3_outliers_missing.ipynb` ค่าผิดปกติและรูปแบบการขาดหายของข้อมูล